# Kanz | كنز — EDA

Goal:
* Explore the data and find the numbers that support our story.

Our story = 5 questions (one per app tab):
```
1 Why care?                      -> the ocean is mostly unknown
2 Where is the treasure?         -> Discover
3 What threatens it?             -> Danger (heat -> bleaching, coral -> fish)
4 Where will we lose it first?   -> At Risk (today's heat + Red Sea)
5 What can we do?                -> Dive & Act (dive sites + rare species)
```

Who does what:
```
Member 1 -> Task 1, Task 2   (Discover)
Member 2 -> Task 3, Task 4   (Danger)
Member 3 -> Task 5, Task 6   (At Risk)
Member 4 -> Task 7           (Dive & Act)
Everyone -> Task 8           (summary)
```

Rules:
* Write the real numbers in every answer (they go to the slides and `docs/decisions.md`)
* Read `Kanz_Build_Plan.md` first → section 2 "Data traps" (important notes about the data)

In [1]:
import os
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

# works inside the project (notebooks/) and inside the shared zip folder
DATA = "../datasets_csv/" if os.path.exists("../datasets_csv") else "datasets_csv/"

# Load the data

All 10 files are loaded here, so the tasks can start directly.

In [2]:
# the 3 biggest files are compressed (.csv.gz) -> pandas reads them the same way

# coral bleaching: missing values are written as "nd"
bleaching = pd.read_csv(DATA + "coral_bleaching_surveys.csv.gz", na_values=["nd"], low_memory=False)

# NOAA reef heat stress
stations = pd.read_csv(DATA + "noaa_reef_stations.csv")
heat = pd.read_csv(DATA + "noaa_reef_heat_stress_daily.csv.gz")

# Reef Life Survey
surveys = pd.read_csv(DATA + "reef_life_surveys.csv")
sites = pd.read_csv(DATA + "reef_life_sites.csv")

# OBIS
animals = pd.read_csv(DATA + "obis_marine_animal_species.csv.gz", low_memory=False)
grid = pd.read_csv(DATA + "ocean_grid_cells.csv")
threatened = pd.read_csv(DATA + "threatened_species_near_dive_sites.csv")

# deep-sea corals + sea temperature trend
corals = pd.read_csv(DATA + "deep_sea_corals_species.csv")
sst_trend = pd.read_csv(DATA + "sea_temperature_anomaly.csv")

In [3]:
# quick overview of every table
tables = {"bleaching": bleaching, "stations": stations, "heat": heat, "surveys": surveys, "sites": sites,
          "animals": animals, "grid": grid, "threatened": threatened,
          "corals": corals, "sst_trend": sst_trend}

overview = pd.DataFrame({
    "rows": [len(df) for df in tables.values()],
    "columns": [df.shape[1] for df in tables.values()],
    "missing %": [round(df.isna().mean().mean() * 100, 1) for df in tables.values()],
}, index=tables.keys())

overview

,rows,columns,missing %
bleaching,41361,62,8.6
stations,214,4,0.0
heat,156220,11,0.0
surveys,36447,9,11.8
sites,4894,12,0.0
animals,164232,11,10.0
grid,1881,12,0.5
threatened,5019,5,0.0
corals,4384,8,8.2
sst_trend,531,6,0.0


In [4]:
bleaching.head()

,Site_ID,Sample_ID,Data_Source,Latitude_Degrees,Longitude_Degrees,Ocean_Name,Reef_ID,Realm_Name,Ecoregion_Name,Country_Name,...,TSA_FrequencyMax,TSA_FrequencyMean,TSA_DHW,TSA_DHW_Standard_Deviation,TSA_DHWMax,TSA_DHWMean,Date,Site_Comments,Sample_Comments,Bleaching_Comments
0,2501,10324336,Donner,23.163,-82.5260,Atlantic,NaN,Tropical Atlantic,Cuba and Cayman Islands,Cuba,...,5.0,0.0,0.00,0.74,7.25,0.18,2005-09-15,NaN,NaN,NaN
1,3467,10324754,Donner,-17.575,-149.7833,Pacific,NaN,Eastern Indo-Pacific,Society Islands French Polynesia,French Polynesia,...,4.0,0.0,0.26,0.67,4.65,0.19,1991-03-15,The bleaching does not appear to have gained ...,The bleaching does not appear to have gained ...,NaN
2,1794,10323866,Donner,18.369,-64.5640,Atlantic,NaN,Tropical Atlantic,Hispaniola Puerto Rico and Lesser Antilles,United Kingdom,...,7.0,0.0,0.00,1.04,11.66,0.26,2006-01-15,NaN,NaN,NaN
3,8647,10328028,Donner,17.760,-64.5680,Atlantic,NaN,Tropical Atlantic,Hispaniola Puerto Rico and Lesser Antilles,United States,...,4.0,0.0,0.00,0.75,5.64,0.20,2006-04-15,NaN,NaN,NaN
4,8648,10328029,Donner,17.769,-64.5830,Atlantic,NaN,Tropical Atlantic,Hispaniola Puerto Rico and Lesser Antilles,United States,...,5.0,0.0,0.00,0.92,6.89,0.25,2006-04-15,NaN,NaN,NaN


In [5]:
surveys.head()

,survey_id,site_code,survey_date,depth,visibility,fish_species,fish_count,fish_biomass,hard_coral_pct
0,1003253,USWC1,2010-07-31,9.0,NaN,8,323,6274.6159,1.0
1,1003254,USWC2,2010-08-16,5.4,NaN,13,146,10759.2408,0.0
2,1003255,USWC3,2010-08-17,11.1,NaN,11,93,13458.6650,1.0
3,1003256,USWC4,2010-08-18,11.1,NaN,10,97,14273.7468,1.0
4,1003257,USWC5,2010-08-19,13.5,NaN,15,153,4541.9844,0.0


In [6]:
grid.head()

,cell_lat,cell_lon,ocean_frac,shelf_frac,depth_mean,depth_max,sst_mean,sst_min,sst_max,sst_range,obis_records,obis_species
0,-80,-180,0.96,0.20,432.500000,697,-0.661721,-1.771484,-0.191777,1.579707,6289,509
1,-80,-175,0.84,0.24,401.857143,663,-0.937450,-1.761230,-0.403809,1.357422,4935,376
2,-80,-170,0.72,0.08,515.500000,1034,-0.901418,-1.772461,-0.489258,1.283203,5061,213
3,-80,-165,0.64,0.00,999.437500,2868,-0.887699,-1.768555,-0.503906,1.264648,4787,234
4,-80,-160,0.52,0.04,1585.615385,4225,-0.908583,-1.740234,-0.490362,1.249873,3291,132


---
# 💎 PART 1 — DISCOVER (Member 1)

# Task 1.1 — Discovery curve

Are we still discovering new ocean animals?

* Keep only species: `taxonRank == "Species"`
* Get the year from `scientificNameAuthorship` (example: "(Linnaeus, 1758)" → 1758)
* Count species per decade and plot a bar chart

Hint:
```
animals["year"] = animals["scientificNameAuthorship"].str.extract(r"(1[789]\d\d|20\d\d)").astype(float)
decade = animals["year"] // 10 * 10
```

In [7]:
#TO-DO

**What did you learn? (2–3 lines with numbers)**

Answer:

# Task 1.2 — Rare and deep treasures

* What % of species were recorded only **once** (`records == 1`)?
* In `corals` (deep-sea corals and sponges): the 10 deepest species (`max_depth_m`)
* How many species are deeper than 1,000 m, and how many have a photo (`ImageURL` not empty)?

In [8]:
#TO-DO

**What did you learn? (2–3 lines with numbers)**

Answer:

# Task 2.1 — More sampling = more species?

Each row of `grid` is a 5° × 5° ocean cell.

* Scatter plot: log(`obis_records`) vs log(`obis_species`)
* Correlation between them

Hint:
```
np.log1p(grid["obis_records"])
```

In [9]:
#TO-DO

**What did you learn? (2–3 lines with numbers)**

Answer:

# Task 2.2 — Where does life live?

* Species vs sea temperature (`sst_mean`) → scatter or groups (cold / mild / warm)
* Species vs depth (`depth_mean`)
* Map: `plt.scatter(grid["cell_lon"], grid["cell_lat"], c=np.log1p(grid["obis_species"]))`

Question: are the richest cells in warm or cold seas?

In [10]:
#TO-DO

**What did you learn? (2–3 lines with numbers)**

Answer:

---
# 🔥 PART 2 — DANGER (Member 2)

# Task 3.1 — Data quality check

* How many rows have a bleaching % (`Percent_Bleaching` not missing)?
* How many unique `Sample_ID`? (each sample is repeated once per seabed type)
* How many rows have `ClimSST == 262.15` (fake −11 °C)?
* Missing values per column (bar chart of the top 10)

In [11]:
#TO-DO

**What did you learn? (2–3 lines with numbers)**

Answer:

# Task 3.2 — Create the target and check balance

Keep one row per sample, then create:
```
0 = none    -> Percent_Bleaching == 0
1 = mild    -> between 0 and 20
2 = severe  -> 20 or more
```
Plot the class balance. Is it balanced?

Hint:
```
df = bleaching[bleaching["Percent_Bleaching"].notna()].drop_duplicates("Sample_ID")
```

In [12]:
#TO-DO

**What did you learn? (2–3 lines with numbers)**

Answer:

# Task 3.3 — Heat stress vs bleaching

`TSA_DHW` = heat stress (degree heating weeks).

* Group it: 0 / 0–2 / 2–4 / 4–8 / 8+
* Share of severe bleaching in each group → bar chart
* Correlation heatmap: heat columns (`TSA_DHW`, `SSTA_DHW`, `TSA`, `SSTA`, `Temperature_Maximum`, ...) with `Percent_Bleaching`

Hint:
```
pd.cut(df["TSA_DHW"], bins=[-0.1, 0, 2, 4, 8, 100], labels=["0", "0-2", "2-4", "4-8", "8+"])
```

In [13]:
#TO-DO

**What did you learn? (2–3 lines with numbers)**

Answer:

# Task 3.4 — Is the ocean warming?

Line chart of `sea_temperature_anomaly_annual` for `entity == "World"` (1850 → today).

In [14]:
#TO-DO

**What did you learn? (2–3 lines with numbers)**

Answer:

# Task 4.1 — Does losing coral mean losing fish?

* Merge `surveys` with `sites` on `site_code` (to get `lat_zone`)
* Box plot: fish species per survey by `lat_zone`
* Group `hard_coral_pct`: 0 / 0–10 / 10–30 / 30–50 / 50+ → average `fish_species` per group (bar chart)

In [15]:
#TO-DO

**What did you learn? (2–3 lines with numbers)**

Answer:

---
# ⚠️ PART 3 — AT RISK (Member 3)

# Task 5.1 — Heat stress today

* Take the **last day** of each station in `heat`
* How many stations have `dhw >= 4` (bleaching alert)?
* Top 10 hottest stations today
* Map: merge with `stations` (lat / lon) and color by `dhw`

Hint:
```
today = heat.groupby("station").tail(1)
```

In [16]:
#TO-DO

**What did you learn? (2–3 lines with numbers)**

Answer:

# Task 5.2 — Heat over time in Saudi waters

Line chart of `dhw` over the last 2 years for:
```
Jazan, Saudi Arabia
Al Madinah and Makkah
Tabuk, Saudi Arabia
```
Add a horizontal line at DHW = 4 and DHW = 8.

Hint:
```
heat["date"] = pd.to_datetime(heat[["year", "month", "day"]])
```

In [17]:
#TO-DO

**What did you learn? (2–3 lines with numbers)**

Answer:

# Task 6.1 — The Red Sea twist

Under strong heat (`TSA_DHW >= 4`), compare the share of **severe** bleaching:
```
Red Sea      -> Ecoregion_Name contains "Red Sea" or "Gulf of Aden"
Rest of world
```
Show the number of surveys for each group too. What does it mean for Saudi reefs?

Hint:
```
df["Ecoregion_Name"].str.contains("Red Sea|Gulf of Aden", na=False)
```

In [18]:
#TO-DO

**What did you learn? (2–3 lines with numbers)**

Answer:

---
# 🤿 PART 4 — DIVE & ACT (Member 4)

# Task 7.1 — Dive sites

* Share of sites per country (top 10) → sampling bias?
* Distribution of `visibility` and `fish_species` per survey
* Top 10 sites by average fish species (merge `surveys` with `sites`)

In [19]:
#TO-DO

**What did you learn? (2–3 lines with numbers)**

Answer:

# Task 7.2 — Rare creatures near dive sites

* Count of threatened species by `category` (CR = critically endangered, EN = endangered, VU = vulnerable)
* The 10 most common threatened species near our dive sites
* Which site has the most CR species?

In [20]:
#TO-DO

**What did you learn? (2–3 lines with numbers)**

Answer:

---
# Task 8 — Summary (everyone)

Write the **5 most important numbers** for our story (one per tab), for example:
```
1 Why care?      -> ...
2 Discover       -> ...
3 Danger         -> ...
4 At Risk        -> ...
5 Dive & Act     -> ...
```

Answer: